# Harmonization: BoT-IoT → ตารางกลาง

สมุดนี้มี mapping และโค้ดสร้างตารางอยู่ใน notebook โดยตรง ตามรูปแบบของ NSL-KDD/UNSW-NB15 ไม่เรียก `run_harmonization()` จากไฟล์ `.py`

อ่านไฟล์ cleaned แยก train/test แล้วสร้าง 3 ตารางที่เชื่อมด้วย `record_id`:

- `features.csv`: คุณลักษณะการเชื่อมต่อ; ไม่มี label
- `labels.csv`: ป้ายกำกับจริงและประเภทการโจมตี
- `metadata.csv`: ต้นทาง split และ flags สำหรับตามรอย

ฟิลด์ที่ dataset นี้ไม่มีจะเก็บเป็น missing พร้อม availability flag ไม่สร้างค่าปลอมขึ้นมา


## 1. กำหนด path และตรวจผล cleansing

หาโฟลเดอร์จากตำแหน่ง notebook และหยุดหากไม่มีไฟล์ cleaned หรือ manifest ที่ขั้นก่อนหน้าควรสร้าง.

In [ ]:
from pathlib import Path

PIPELINE_DIR = next(
    candidate
    for ancestor in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    for candidate in (ancestor, ancestor / "data-pipeline")
    if (candidate / "CENTRAL" / "feature_contract.json").is_file()
)

def require(condition, message):
    if not condition:
        raise ValueError(message)


def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''):h.update(block)
    return h.hexdigest()


def raw_manifest(dataset_dir):
    return [{'file':p.relative_to(Path(dataset_dir)/'raw').as_posix(),'bytes':p.stat().st_size,'sha256':digest(p)} for p in sorted((Path(dataset_dir)/'raw').rglob('*')) if p.is_file() and p.suffix.lower() in {'.csv','.gz','.txt'}]


def verify_manifest(dataset_dir, manifest):
    saved=pd.read_csv(manifest).sort_values('file').reset_index(drop=True)
    now=pd.DataFrame(raw_manifest(dataset_dir)).sort_values('file').reset_index(drop=True)
    require(len(now)>0 and saved[['file','sha256']].equals(now[['file','sha256']]),f'Stale raw manifest: {manifest}; rerun cleansing/harmonization')

verify_manifest(DATASET_DIR, clean_manifest_path)


## 2.1 เตรียมกฎปรับหมวดหมู่และตรวจคุณภาพ

กำหนดชื่อ protocol, service และ connection state ที่ใช้ร่วมกัน พร้อมกฎทำเครื่องหมายแถวที่มีข้อมูลขัดกัน


In [ ]:
CLASS_NAMES = [
    "Benign / Normal",
    "Denial of Service (DoS)",
    "Reconnaissance & Probing",
    "Exploits & PrivEsc",
    "Credential Abuse (R2L)",
    "Malware, C2 & Botnet",
]


SERVICE_ALIASES = {
    'http':'http', 'www':'http', 'http_443':'https_ssl',
    'https':'https_ssl', 'ssl':'https_ssl',
    'ftp':'ftp', 'ftp_data':'ftp', 'ftp-data':'ftp',
    'ssh':'ssh', 'smtp':'smtp', 'domain':'dns', 'domain_u':'dns', 'dns':'dns',
}


STATE_ALIASES = {
    'sf':'established', 's1':'established', 's2':'established', 's3':'established',
    'fin':'established', 'con':'established', 'established':'established',
    's0':'no-reply', 'req':'no-reply', 'int':'no-reply', 'no-reply':'no-reply',
    'rej':'reset', 'rsto':'reset', 'rstr':'reset', 'rstos0':'reset',
    'rst':'reset', 'reset':'reset',
}


QUALITY_FLAGS = ['invalid_required_value', 'bytes_packet_conflict', 'state_counter_conflict']


def normalize_boolean_metadata(metadata):
    columns=[c for c in metadata if c.startswith(('has_','invalid_')) or c in QUALITY_FLAGS+['quality_ok','service_identified','state_reclassified','dedup_flag','is_kddtest21_view']]
    for col in columns:
        values=metadata[col].astype('string').str.lower()
        if not values.isin(['true','false','1','0']).all():raise ValueError(f'Invalid boolean metadata: {col}')
        metadata[col]=values.map({'true':True,'false':False,'1':True,'0':False}).astype(bool)
    return metadata


def text(values):
    return values.astype('string').str.strip().str.lower().replace({'':'', '-':'-'})


def normalize_protocol(values):
    values=text(values).replace({'6':'tcp','6.0':'tcp','17':'udp','17.0':'udp',
                                 '1':'icmp','1.0':'icmp','58':'icmp','58.0':'icmp','ipv6-icmp':'icmp'})
    present=values.notna() & ~values.isin(['','-'])
    return values.where(values.isin(['tcp','udp','icmp']),'other').where(present,pd.NA)


def port_to_service(value):
    if pd.isna(value) or str(value).strip()=='':return 'other'
    token=str(value).strip()
    try:
        number=int(token,16) if token.lower().startswith('0x') else float(token)
    except (ValueError,TypeError) as error:
        raise ValueError(f'Invalid destination port: {value!r}') from error
    if not np.isfinite(number) or number!=int(number) or not 0<=number<=65535:
        raise ValueError(f'Invalid destination port: {value!r}')
    port=int(number)
    return {53:'dns',20:'ftp',21:'ftp',80:'http',8080:'http',443:'https_ssl',
            25:'smtp',465:'smtp',587:'smtp',22:'ssh'}.get(port,'other')


def quality_flags(features, metadata):
    f=features
    invalid=pd.Series(False,index=f.index)
    for col in ['duration','bytes_forward','bytes_backward']:
        values=pd.to_numeric(f[col],errors='coerce')
        invalid |= values.isna() | ~np.isfinite(values) | values.lt(0)
    invalid |= f.protocol.isna() | f.protocol.astype('string').str.strip().isin(['','-'])
    packet_conflict=pd.Series(False,index=f.index)
    for direction in ['forward','backward']:
        col='packets_'+direction
        if col in f:
            values=pd.to_numeric(f[col],errors='coerce')
            packet_conflict |= values.eq(0) & f['bytes_'+direction].gt(0)
            invalid |= f[col].notna() & (~np.isfinite(values) | values.lt(0))
    raw_state=text(metadata.get('source_connection_state',pd.Series(pd.NA,index=f.index,dtype='string')))
    has_response=f.bytes_backward.gt(0)
    if 'packets_backward' in f:has_response |= f.packets_backward.gt(0)
    # REQ/INT can have response traffic. Literal no-reply/S0 with a response is
    # contradictory source evidence; do not silently erase that evidence.
    state_conflict=raw_state.isin(['no-reply','s0']) & has_response
    return pd.DataFrame({'invalid_required_value':invalid,
                         'bytes_packet_conflict':packet_conflict,
                         'state_counter_conflict':state_conflict},index=f.index)


## 2.2 กำหนดการแปลง label เป็นคลาสกลาง

จับคู่ label ต้นทางกับรหัสคลาสและตรวจว่าตรงกับ label แบบปกติหรือโจมตี


In [ ]:
TACTICS = ['Baseline Operations','TA0040: Impact','TA0043/TA0007: Recon & Discovery','TA0001/TA0004: Initial Access & PrivEsc','TA0006: Credential Access','TA0011/TA0008: C2 & Lateral Movement']


COMMON_MAPPING = {'normal':0,'benign':0,'dos':1,'dos/ddos':1,'ddos':1,'probe':2,'probe/recon':2,'reconnaissance':2,'analysis':2,'exploits':3,'exploit/u2r':3,'shellcode':3,'u2r':3,'r2l':4,'brute-force/r2l':4,'generic':5,'generic/fuzzer':5,'fuzzers':5,'backdoor':5,'backdoors':5,'botnet/backdoor':5,'worms':5}


SOURCE_MAPPING = {
    'NSL-KDD': {k:COMMON_MAPPING[k] for k in ['normal','dos','probe','r2l','u2r']},
    'UNSW-NB15': {k:COMMON_MAPPING[k] for k in ['normal','analysis','backdoor','backdoors','dos','exploits','fuzzers','generic','reconnaissance','shellcode','worms']},
    'CIC-IDS2017': {**COMMON_MAPPING, 'dos hulk':1,'dos goldeneye':1,'dos slowloris':1,'dos slowhttptest':1,'heartbleed':3,'portscan':2,'ftp-patator':4,'ssh-patator':4,'web attack - brute force':4,'web attack - xss':3,'web attack - sql injection':3,'web attack brute force':4,'web attack xss':3,'web attack sql injection':3,'infiltration':3,'bot':5},
    # Project-level category grouping, not per-record MITRE attribution.
    'BoT-IoT': {**COMMON_MAPPING,'theft':5,'data_theft':5,'bruteforce':4},
}


def map_labels(labels, metadata):
    result=labels.copy(); ids=pd.Series(index=labels.index,dtype='int64')
    for source,index in metadata.groupby('source_dataset').groups.items():
        require(source in SOURCE_MAPPING,f'Unsupported source {source}')
        values=labels.loc[index,'label_source_category'].astype('string').str.strip().str.lower().str.replace(r'\s+',' ',regex=True)
        mapped=values.map(SOURCE_MAPPING[source])
        require(mapped.notna().all(),f'Unmapped labels in {source}: {values[mapped.isna()].value_counts().to_dict()}')
        ids.loc[index]=mapped.astype(int)
    ids=ids.astype(int)
    require(labels.label_binary.eq(ids.ne(0).astype(int)).all(),'Mapped categories contradict original binary truth')
    result['attack_category_id']=ids;result['attack_category']=ids.map(dict(enumerate(CLASS_NAMES)));result['mitre_tactic']=ids.map(dict(enumerate(TACTICS)))
    return result


## 2.3 เตรียมฟังก์ชัน harmonization กลาง

กำหนดขั้นนำกฎกลางไปใช้กับตารางทั้งสาม รวมถึงข้อมูลกำกับว่าแต่ละค่ามาจากอะไร


In [ ]:
def harmonize_tables(dataset, features, labels, metadata, source):
    f=features.reset_index(drop=True).copy()
    l=labels.reset_index(drop=True).copy()
    m=metadata.reset_index(drop=True).copy()
    source=source.reset_index(drop=True)
    if not len(source)==len(f)==len(l)==len(m):raise ValueError('Source/output row count mismatch')
    for col in ['packets_forward','packets_backward']:
        if col not in f:f[col]=np.nan
    protocol_col={'NSL-KDD':'protocol_type','UNSW-NB15':'proto','CIC-IDS2017':'protocol','BoT-IoT':'proto'}[dataset]
    state_col={'NSL-KDD':'flag','UNSW-NB15':'state','CIC-IDS2017':'flow_state','BoT-IoT':'state'}[dataset]
    m['source_protocol']=source[protocol_col].astype('string')
    m['source_connection_state']=source[state_col].astype('string')
    f['protocol']=normalize_protocol(source[protocol_col])
    if dataset in ['NSL-KDD','UNSW-NB15']:
        service=text(source['service'])
        present=service.notna() & ~service.isin(['','-'])
        f['service_group']=service.map(SERVICE_ALIASES).fillna('other')
        m['source_service']=source['service'].astype('string')
        m['service_basis']='source_service_name'
        m['service_identified']=present
        m['has_service']=present
    else:
        port='dest_port' if dataset=='CIC-IDS2017' else 'dport'
        values=source[port] if port in source else pd.Series(pd.NA,index=source.index,dtype='string')
        m['source_service']=values.astype('string')
        if dataset=='BoT-IoT':
            # ARP has no transport port. The mirror represents it as -1.
            missing_arp_port=text(source[protocol_col]).eq('arp') & text(values).isin(['-1','-1.0'])
            values=values.mask(missing_arp_port.fillna(False))
        present=values.notna() & values.astype('string').str.strip().ne('')
        f['service_group']=values.map(port_to_service)
        m['service_basis']=np.where(present,'destination_port_hint','unavailable')
        m['service_identified']=False  # Port 443 is a hint, not proof of TLS.
        m['has_service']=present
    state=text(source[state_col])
    present_state=state.notna() & ~state.isin(['','-'])
    base=state.map(STATE_ALIASES).fillna('other')
    response=f.bytes_backward.gt(0)
    if 'packets_backward' in f:response |= f.packets_backward.gt(0)
    f['conn_state_group']=base.mask(base.eq('no-reply') & response,'other')
    m['state_reclassified']=base.ne(f.conn_state_group)
    m['connection_state_basis']='source_state_and_directional_counters'
    m['has_connection_state']=present_state
    m['has_duration']=f.duration.notna()
    m['has_protocol']=f.protocol.notna()
    m['has_directional_bytes']=f[['bytes_forward','bytes_backward']].notna().all(axis=1)
    m['has_packet_stats']=f[['packets_forward','packets_backward']].notna().all(axis=1)
    for col,out in [('duration','invalid_duration'),('bytes_forward','invalid_bytes_forward'),('bytes_backward','invalid_bytes_backward')]:
        values=pd.to_numeric(f[col],errors='coerce')
        m[out]=values.isna() | ~np.isfinite(values) | values.lt(0)
    flags=quality_flags(f,m)
    for col in QUALITY_FLAGS:m[col]=flags[col]
    m['quality_ok']=~flags.any(axis=1)
    m['quality_issues']=[';'.join(c for c in QUALITY_FLAGS if row[c]) for row in flags.to_dict('records')]
    m['duration_unit']='seconds'
    m=normalize_boolean_metadata(m)
    l['source_attack_category']=l.attack_category.astype('string')
    l=map_labels(l,m)  # One taxonomy for source tables and CENTRAL.
    return f,l,m


def map_state_to_group(state_value: str) -> str:
    state = str(state_value).upper()
    if state in ("CON", "FIN", "ESTABLISHED"):
        return "established"
    if state in ("REQ", "INT", "NO-REPLY"):
        return "no-reply"
    if state in ("RST", "REJ", "RESET"):
        return "reset"
    return "other"

def normalize_category(values: pd.Series, allowed: list[str]) -> pd.Series:
    lowered = values.astype(str).str.lower()
    return lowered.where(lowered.isin(allowed), "other")


## 3. แปลงข้อมูลทีละ split เป็น features / labels / metadata

ตัวอย่าง mapping: `sbytes` หรือ `total_fwd_bytes` → `bytes_forward`; label แยกไว้ในตาราง labels.

In [ ]:
feature_parts, label_parts, metadata_parts = [], [], []
source_frames = []

for split_name, filename in RAW_FILES.items():
    frame = pd.read_csv(CLEAN_DIR / filename, low_memory=False)
    source_frames.append(frame.copy())
    record_ids = [f"{DATASET_NAME}:{split_name}:{row_id}" for row_id in frame["orig_row_id"]]

    # BoT-IoT field names -> shared flow fields
    connection_state = frame["state"].apply(map_state_to_group)
    protocol = normalize_protocol(frame["proto"])
    duration = frame["dur"].astype(float)
    bytes_forward = frame["sbytes"].astype(float)
    bytes_backward = frame["dbytes"].astype(float)
    packets_forward = frame["spkts"].astype(float)
    packets_backward = frame["dpkts"].astype(float)
    packet_total = packets_forward + packets_backward
    mean_packet_size = np.where(packet_total > 0, (bytes_forward + bytes_backward) / packet_total, np.nan)

    feature_parts.append(pd.DataFrame({
        "record_id": record_ids,
        "duration": duration,
        "protocol": protocol,
        "bytes_forward": bytes_forward,
        "bytes_backward": bytes_backward,
        "conn_state_group": connection_state,
        "packets_forward": packets_forward,
        "packets_backward": packets_backward,
        "mean_packet_size": mean_packet_size,
        "ttl_forward": np.nan,
        "ttl_backward": np.nan,
        "tcp_window_forward": np.nan,
        "tcp_window_backward": np.nan,
        "nsl_hot": np.nan,
        "nsl_logged_in": np.nan,
        "nsl_failed_logins": np.nan,
        "nsl_root_shell": np.nan,
        "nsl_count_2s": np.nan,
        "nsl_srv_count_2s": np.nan,
        "nsl_same_srv_rate_2s": np.nan,
        "nsl_serror_rate_2s": np.nan,
    }))

    source_category = frame["category"].astype(str)
    label_parts.append(pd.DataFrame({
        "record_id": record_ids,
        "label_raw": frame["subcategory"].astype(str),
        "label_source_category": source_category,
        "attack_category": source_category,
        "label_binary": frame["attack"].astype(int),
    }))

    metadata_parts.append(pd.DataFrame({
        "record_id": record_ids,
        "source_dataset": DATASET_NAME,
        "orig_row_id": frame["orig_row_id"],
        "source_split": split_name,
        "capture_time": pd.NA,
        "split_group": split_name,
        "dedup_flag": frame["_duplicate_flag"],
        "sample_weight": 1.0,
        "has_duration": True,
        "invalid_duration": False,
        "has_protocol": frame["proto"].notna().all(),
        "has_directional_bytes": True,
        "invalid_bytes_forward": False,
        "invalid_bytes_backward": False,
        "has_capture_time": False,
        "has_service": "dport" in frame and frame["dport"].notna().all(),
        "has_connection_state": frame["state"].notna().all(),
        "has_packet_stats": True,
        "has_tcp_details": False,
        "has_nsl_content": False,
        "has_nsl_window": False,
        "is_kddtest21_view": False,
    }))


## 4. ตรวจ ID และความสอดคล้องก่อนบันทึก

ยืนยันว่า ID ไม่ซ้ำและเรียงตรงกันทั้งสามตาราง ก่อน export.

In [ ]:
features = pd.concat(feature_parts, ignore_index=True)
labels = pd.concat(label_parts, ignore_index=True)
metadata = pd.concat(metadata_parts, ignore_index=True)
features, labels, metadata = harmonize_tables(DATASET_NAME, features, labels, metadata, pd.concat(source_frames, ignore_index=True))

if not (features["record_id"].is_unique and labels["record_id"].is_unique and metadata["record_id"].is_unique):
    raise ValueError("record_id ซ้ำในตาราง harmonized")
if not (features["record_id"].tolist() == labels["record_id"].tolist() == metadata["record_id"].tolist()):
    raise ValueError("record_id เรียงไม่ตรงกันใน features, labels และ metadata")

print("Feature table:", features.shape)
print("Label table:", labels.shape)
print("Metadata table:", metadata.shape)
display(labels["attack_category"].value_counts(dropna=False).rename("rows").to_frame())


## 5. บันทึกผลและ provenance

In [ ]:
tables = {"features": features, "labels": labels, "metadata": metadata}
for table_name, table in tables.items():
    table.to_csv(HARM_DIR / f"{table_name}.csv", index=False)

report = pd.DataFrame([
    {"table": name, "rows": len(table), "columns": len(table.columns)}
    for name, table in tables.items()
])
report.to_csv(HARM_DIR / "harmonization_report.csv", index=False)

quality = metadata.groupby("source_split")[["invalid_duration", "invalid_bytes_forward", "invalid_bytes_backward", "bytes_packet_conflict", "state_counter_conflict"]].sum().reset_index()
quality.to_csv(HARM_DIR / "harmonization_quality.csv", index=False)
pd.read_csv(clean_manifest_path).to_csv(HARM_DIR / "source_manifest.csv", index=False)

provenance = {
    "dataset": DATASET_NAME,
    "pipeline_version": "quad_v1_notebook",
    "features_rows": len(features),
    "labels_rows": len(labels),
    "metadata_rows": len(metadata),
    "tables_verified": False,
}
(HARM_DIR / "provenance.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")
print("Saved files:")
for path in sorted(HARM_DIR.iterdir()):
    print(f"- {path.name}: {path.stat().st_size:,} bytes")


## 6. เตรียมรายงานคุณภาพ

กำหนดโครงสร้างรายงานคุณภาพและวิธีเขียนตารางแยกแถวที่ถูกกักไว้


In [ ]:
PROJECT_ROOT = PIPELINE_DIR.parent

NUMERIC = ["duration", "bytes_forward", "bytes_backward"]


CATEGORIES = {
    "protocol": ["tcp", "udp", "icmp", "other"],
    "service_group": ["dns", "ftp", "http", "https_ssl", "smtp", "ssh", "other"],
    "conn_state_group": ["established", "no-reply", "reset", "other"],
}


CORE = NUMERIC + list(CATEGORIES)


def write_quality_outputs(dataset_dir, f,l,m):
    from pathlib import Path
    d=Path(dataset_dir)/'output/harmonized'
    bad=~m.quality_ok
    q=d/'quarantine';q.mkdir(parents=True,exist_ok=True)
    for name,frame in [('features',f),('labels',l),('metadata',m)]:
        frame.loc[bad].to_csv(q/(name+'.csv'),index=False)
    rows=[]
    for split,block in m.groupby('source_split'):
        row={'source_dataset':dataset_dir.name,'split':split,'rows':len(block),
             'eligible_rows':int(block.quality_ok.sum()),'quarantined_rows':int((~block.quality_ok).sum())}
        for col in QUALITY_FLAGS+['invalid_duration','invalid_bytes_forward','invalid_bytes_backward','state_reclassified']:
            row[col]=int(block[col].sum())
        rows.append(row)
    pd.DataFrame(rows).to_csv(d/'harmonization_quality.csv',index=False)
    path=d/'provenance.json'
    import json
    provenance=json.loads(path.read_text(encoding='utf8')) if path.exists() else {}
    provenance.update({'harmonization_schema_version':6,'tables_verified':True,
                       'eligible_rows':int((~bad).sum()),'quarantined_rows':int(bad.sum()),
                       'quality_policy':'Retain all source records; exclude flagged records from CENTRAL and keep separate quarantine tables'})
    write_json(path,provenance)


def write_json(path, value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(value,indent=2,ensure_ascii=False,default=lambda x:x.item() if hasattr(x,'item') else str(x)),encoding='utf8')


## 7. เตรียมการตรวจผลลัพธ์และ dependency stamp

กำหนดวิธีอ่านตารางกลับมาตรวจและเก็บ checksum ของไฟล์ที่ใช้


In [ ]:
def read_tables(folder):
    # Source labels can be numeric (UNSW) or text (NSL/CIC). Preserve lexical
    # identifiers consistently across source tables, concatenation and CSV reload.
    types={name:'string' for name in ['record_id','label_raw','label_source_category','source_attack_category','orig_row_id','split_group','source_dataset','source_split','source_protocol','source_connection_state','source_service','service_basis','connection_state_basis','duration_unit','quality_issues','capture_time']}
    f,l,m=tuple(pd.read_csv(Path(folder)/(t+'.csv'),low_memory=False,dtype=types) for t in ['features','labels','metadata'])
    return f,l,normalize_boolean_metadata(m)


def validate_tables(f,l,m,allow_quarantined=False):
    require(len(f)>0 and len(f)==len(l)==len(m),'Empty tables or row-count mismatch')
    for name,frame in [('features',f),('labels',l),('metadata',m)]:
        require('record_id' in frame and frame.record_id.notna().all() and frame.record_id.astype(str).str.strip().ne('').all() and frame.record_id.is_unique,f'{name}: missing/duplicate record_id')
    require(f.record_id.equals(l.record_id) and f.record_id.equals(m.record_id),'Record IDs/order mismatch')
    require(not set(f.columns).intersection({'label','label_raw','label_source_category','attack_category','attack_category_id','label_binary','mitre_tactic','is_attack','source_dataset','sample_weight','data_split','is_synthetic'}),'Target/metadata leakage in features')
    require(set(CORE).issubset(f.columns),'Missing core columns')
    computed=quality_flags(f,m)
    eligible=~computed.any(axis=1)
    if 'quality_ok' in m:
        require(m.quality_ok.notna().all() and m.quality_ok.eq(eligible).all(),'quality_ok disagrees with recomputed quality rules')
        for col in computed:require(col in m and m[col].eq(computed[col]).all(),f'{col}: quality flag mismatch')
    require(allow_quarantined or eligible.all(),'Invalid duration/bytes/protocol or cross-field conflicts: quarantined records are not allowed in model splits')
    checked=f.loc[eligible] if allow_quarantined else f
    for c in NUMERIC:
        z=pd.to_numeric(checked[c],errors='coerce')
        require(np.isfinite(z).all() and z.ge(0).all(),f'{c}: missing, infinite or negative input')
    for c,values in CATEGORIES.items():require(checked[c].notna().all() and checked[c].isin(values).all(),f'{c}: missing/unknown categories')
    response=checked.bytes_backward.gt(0)
    if 'packets_backward' in checked:response |= checked.packets_backward.gt(0)
    require(not (checked.conn_state_group.eq('no-reply') & response).any(),'no-reply contradicts backward traffic')
    for c in ['packets_forward','packets_backward','mean_packet_size','ttl_forward','ttl_backward','tcp_window_forward','tcp_window_backward']:
        if c in checked:
            z=checked[c].dropna()
            require(np.isfinite(z).all() and z.ge(0).all(),f'{c}: invalid measured extension')
    require('label_binary' in l and l.label_binary.isin([0,1]).all(),'Invalid binary labels')
    if 'attack_category_id' in l:
        require(l.attack_category_id.isin(range(6)).all(),'Invalid central classes')
        require(l.label_binary.eq(l.attack_category_id.ne(0).astype(int)).all(),'Binary/multiclass labels disagree')
        require(l.attack_category.eq(l.attack_category_id.map(dict(enumerate(CLASS_NAMES)))).all(),'Class name/ID mismatch')
    if 'sample_weight' in m:require(np.isfinite(m.sample_weight).all() and m.sample_weight.gt(0).all(),'Invalid sample weights')
    for flag in ['invalid_duration','invalid_bytes_forward','invalid_bytes_backward']:
        if flag in m:require(not m.loc[eligible,flag].astype(str).str.lower().isin(['true','1']).any(),f'{flag}: flagged invalid rows')


def code_signature(pipeline_dir):
    p=Path(pipeline_dir)
    files=sorted([*p.glob('*.py'),*p.glob('CENTRAL/*.json'),*p.glob('CENTRAL/*.ipynb'),*p.glob('sub-dataset/*/*.ipynb'),*p.glob('sub-dataset/*/*.py')])
    signature = {f.relative_to(p).as_posix():digest(f) for f in files}
    shared = PROJECT_ROOT / "webapp/Dashboard/backend/src/models/central_contract.py"
    signature["../webapp/Dashboard/backend/src/models/central_contract.py"] = digest(shared)
    return signature


def stamp_dataset(dataset_dir):
    d=Path(dataset_dir);p=d.parent.parent
    f,l,m=read_tables(d/'output/harmonized');validate_tables(f,l,m,allow_quarantined=True)
    verify_manifest(d,d/'output/cleansing/source_manifest.csv')
    verify_manifest(d,d/'output/harmonized/source_manifest.csv')
    cleaned=cleaned_fingerprints(d)
    if 'quality_ok' in m:write_quality_outputs(d,f,l,m)
    origin=d/'raw/source_provenance.json'
    write_json(d/'output/harmonized/dependencies.json',{'raw':raw_manifest(d),'code':code_signature(p),'cleaned':cleaned,'source_provenance':digest(origin) if origin.exists() else None,'tables':{t:digest(d/'output/harmonized'/f'{t}.csv') for t in ['features','labels','metadata']}})


def cleaned_fingerprints(dataset_dir):
    d=Path(dataset_dir)
    names={'NSL-KDD':['KDDTrain_clean.csv','KDDTest_clean.csv','KDDTest-21_clean.csv'],
           'UNSW-NB15':['UNSW_NB15_training-set.csv','UNSW_NB15_testing-set.csv'],
           'CIC-IDS2017':['CIC_IDS2017_training-set.csv','CIC_IDS2017_testing-set.csv'],
           'BoT-IoT':['BoT_IoT_training-set.csv','BoT_IoT_testing-set.csv']}[d.name]
    for name in names:require((d/'output/cleansing'/name).is_file(),f'Missing cleaned file: {d.name}/{name}')
    return {name:digest(d/'output/cleansing'/name) for name in names}


## 8. ตรวจไฟล์และบันทึกผลยืนยัน

ตรวจไฟล์ที่เพิ่งเขียน สร้างรายงานคุณภาพ และบันทึก dependencies.json ให้ CENTRAL ใช้ตรวจที่มา


In [ ]:
stamp_dataset(DATASET_DIR)
